# 01 — Extraction Accuracy Eval

Verifies `extraction.extract_form()` against known ground truth for the four
mock forms in `data/sample_forms/` (see APPROACH.md §3.2, §6 step 2).

Ground truth values below are copied from `_generate_samples.py`, which
generated these forms — so this is a controlled, deterministic check, not a
held-out test set. It answers one question: **does schema-guided extraction
correctly pull every structured field**, including from the OCR'd scanned
form (`hospital_002_scanned`), which is the harder case.

Requires Ollama running locally with the extraction model pulled
(`ollama pull llama3.1:8b` or whatever `llm_client.DEFAULT_MODEL` points to).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
import ingestion
import extraction
import llm_client

SAMPLE_DIR = os.path.abspath("../data/sample_forms")

if not llm_client.is_available():
    print(f"⚠️  Ollama / {llm_client.DEFAULT_MODEL} not reachable.")
    print("   Run `ollama serve` and `ollama pull` the model before running this notebook.")
else:
    print("✅ Ollama reachable, model available.")

✅ Ollama reachable, model available.


## Ground truth (from `_generate_samples.py`)

In [2]:
GROUND_TRUTH = {
    "membership_001": {
        "name": "Ananya Rao",
        "date_of_birth": "14-03-1994",
        "email": "ananya.rao@example.com",
        "phone": "+91-9876543210",
        "occupation": "Software Engineer",
        "application_date": "02-01-2026",
        "status": "Approved",
    },
    "membership_002": {
        "name": "Vikram Nair",
        "date_of_birth": "22-07-1988",
        "email": "vikram.nair@example.com",
        "phone": "+91-9812345678",
        "occupation": "Freelance Designer",
        "application_date": "15-01-2026",
        "status": "Rejected",
    },
    "hospital_001": {
        "patient_name": "Rahul Menon",
        "patient_id": "H-2026-0091",
        "date_of_birth": "09-11-1975",
        "doctor": "Dr. Lakshmi Iyer",
        "department": "Cardiology",
        "admission_date": "10-02-2026",
        "discharge_date": "15-02-2026",
        "diagnosis": "Acute myocardial infarction",
    },
    "hospital_002_scanned": {
        "patient_name": "Fathima Sheikh",
        "patient_id": "H-2026-0114",
        "date_of_birth": "03-05-1990",
        "doctor": "Dr. Arjun Kapoor",
        "department": "Orthopedics",
        "admission_date": "20-02-2026",
        "discharge_date": "24-02-2026",
        "diagnosis": "Fractured tibia (left leg)",
    },
}

# Free-text fields are excluded from exact-match ground truth above (remarks /
# doctors_notes are full paragraphs — the LLM rephrasing them slightly is fine,
# what matters is the structured fields match exactly). We spot-check the
# free-text field separately below with a keyword containment check instead.
FREE_TEXT_KEYWORDS = {
    "membership_001": ["credit history", "premium"],
    "membership_002": ["defaults", "reapply"],
    "hospital_001": ["angioplasty", "cardiac rehab"],
    "hospital_002_scanned": ["fixation", "physiotherapy"],
}

## Run extraction on every sample form

In [3]:
raw_forms = ingestion.ingest_directory(SAMPLE_DIR)
print(f"Ingested {len(raw_forms)} form(s): {[f.form_id for f in raw_forms]}")

results = {}
for raw in raw_forms:
    try:
        results[raw.form_id] = extraction.extract_form(raw)
        r = results[raw.form_id]
        print(f"{raw.form_id}: type={r.form_type} "
              f"(detected via {r.detection_method}, {r.extraction_attempts} attempt(s))")
    except extraction.ExtractionError as e:
        print(f"{raw.form_id}: FAILED — {e}")

Ingested 4 form(s): ['hospital_001', 'hospital_002_scanned', 'membership_001', 'membership_002']
hospital_001: type=hospital (detected via keyword, 1 attempt(s))
hospital_002_scanned: type=hospital (detected via keyword, 1 attempt(s))
membership_001: type=membership (detected via keyword, 1 attempt(s))
membership_002: type=membership (detected via keyword, 1 attempt(s))


## Score structured fields against ground truth

In [4]:
rows = []
free_text_field = {"membership": "remarks", "hospital": "doctors_notes"}

for form_id, expected in GROUND_TRUTH.items():
    if form_id not in results:
        rows.append({"form_id": form_id, "field": "(entire form)", "match": False,
                      "expected": "extracted", "actual": "EXTRACTION FAILED"})
        continue

    extracted = results[form_id].data.model_dump()
    for field_name, expected_value in expected.items():
        actual_value = extracted.get(field_name)
        rows.append({
            "form_id": form_id,
            "field": field_name,
            "match": actual_value == expected_value,
            "expected": expected_value,
            "actual": actual_value,
        })

    # Free-text spot check: keyword containment, not exact match.
    ft_field = free_text_field[results[form_id].form_type]
    ft_text = (extracted.get(ft_field) or "").lower()
    for kw in FREE_TEXT_KEYWORDS.get(form_id, []):
        rows.append({
            "form_id": form_id,
            "field": f"{ft_field} (contains {kw!r})",
            "match": kw.lower() in ft_text,
            "expected": f"contains {kw!r}",
            "actual": ft_text[:80] + "..." if len(ft_text) > 80 else ft_text,
        })

df = pd.DataFrame(rows)
df

,form_id,field,match,expected,actual
0,membership_001,name,True,Ananya Rao,Ananya Rao
1,membership_001,date_of_birth,True,14-03-1994,14-03-1994
2,membership_001,email,True,ananya.rao@example.com,ananya.rao@example.com
3,membership_001,phone,True,+91-9876543210,+91-9876543210
4,membership_001,occupation,True,Software Engineer,Software Engineer
5,membership_001,application_date,True,02-01-2026,02-01-2026
6,membership_001,status,True,Approved,Approved
7,membership_001,remarks (contains 'credit history'),True,contains 'credit history',applicant has a strong credit history and was ...
8,membership_001,remarks (contains 'premium'),True,contains 'premium',applicant has a strong credit history and was ...
9,membership_002,name,True,Vikram Nair,Vikram Nair


In [5]:
# Summary: per-form and overall accuracy
per_form = df.groupby("form_id")["match"].mean().rename("field_match_rate")
print(per_form)
print()
print(f"Overall field match rate: {df['match'].mean():.1%} "
      f"({df['match'].sum()}/{len(df)} fields)")

failures = df[~df["match"]]
if len(failures):
    print("\nMismatches to inspect:")
    print(failures[["form_id", "field", "expected", "actual"]].to_string(index=False))
else:
    print("\nNo mismatches — extraction is fully aligned with ground truth.")

form_id
hospital_001            1.0
hospital_002_scanned    1.0
membership_001          1.0
membership_002          1.0
Name: field_match_rate, dtype: float64

Overall field match rate: 100.0% (38/38 fields)

No mismatches — extraction is fully aligned with ground truth.


## Notes

- `hospital_002_scanned` is the one form that went through the OCR fallback
  (§3.1) before extraction — if any form is going to mismatch, it'll likely
  be this one, since Tesseract can introduce small character errors (e.g.
  `l` vs `1`, spacing around `+91-`) that a text-based PDF wouldn't have.
- A mismatch doesn't necessarily mean the extraction prompt is wrong — check
  whether the *raw ingested text* itself already had the error
  (`ingestion.ingest_file(path).text`) before assuming the LLM's fault.
- If mismatches cluster on one field across multiple forms (e.g. `phone`
  formatting), that's a signal to tighten `schemas.py`'s field description
  or `extraction.py`'s prompt for that field specifically, rather than
  raising `MAX_EXTRACTION_RETRIES` — a smaller, more specific prompt fix is
  cheaper than an extra retry per form.